# Time Reduction functions and calculate the rations between hashing to see which one is faster

## Imports

In [1]:
from hashlib import sha256
import mmh3
from math import e
import random
import numpy as np
import time
import pandas as pd
from tqdm import tqdm

## Constants

In [2]:
N = 2 ** 16  # keyspace
p = 1 - e ** -2  # our table coverage - 86%
t = 80  # number of columns

## Helper Functions

In [3]:
# Hash function
def H(x):
    return int.from_bytes(sha256(x.to_bytes(8)).digest())

def H_c(x):
    return sha256(x.to_bytes(8)).digest()  # return bytes directly
    
# mmh reduction function
def r_mmh(N, t, y, i, ell=0):
    return mmh3.hash(y, i + (ell*t), signed=False) % N

def r(N, t, y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	# return (y + i + ell*t) % N
    # if everything is bytes

    return (y + i + ell*t) & (N - 1) # N is a power of 2 so can use bitwise AND instead of modulo

## Time Hash Function

In [4]:
## Create a list of 10,000 list, each of which has 10,000 points to use in both experiments to make it fair
points_list = []
for i in range(10000):
    points_list.append(random.sample(range(N), 10000))

KeyboardInterrupt: 

In [ ]:
## for the first hash function

# store times
hash_int_times = []

# for 10,000 trials
for i in range(10000):
    # choose 10,000 inputs and hash them
    points = points_list[i]

    # start timer
    start_time = time.perf_counter()

    # hash all points
    for i in points:
        H(i)

    # stop timer
    hash_time = time.perf_counter() - start_time

    # add to list
    hash_int_times.append(hash_time)

# make a df 
hash_df = pd.DataFrame({"int times" : hash_int_times})

#####################################################################################

## for the second hash function

# store times
hash_bytes_times = []

# for 10,000 trials
for i in range(10000):
    # choose 10,000 inputs and hash them
    points = points_list[i]

    # start timer
    start_time = time.perf_counter()

    # hash all points
    for i in points:
        H_c(i)

    # stop timer
    hash_time = time.perf_counter() - start_time

    # add to list
    hash_bytes_times.append(hash_time)

# make a df 
hash_df["bytes times"] = hash_bytes_times

In [ ]:
## describe the dataframe
hash_df.describe()

,int times,bytes times
count,10000.000000,10000.000000
mean,0.013217,0.010955
std,0.001219,0.001674
min,0.012002,0.009849
25%,0.012630,0.010304
50%,0.012951,0.010602
75%,0.013409,0.011061
max,0.079819,0.052737


## Set up parameters to test reduction functions
- hash table containing 10,000 points
- 10,000 K values

In [ ]:
# choose random unique 100,000 inputs and hash them
points = random.sample(range(N), 10000)
    
# hashes are integers
hash_int = []
for i in points:
    hash_int.append(H(i))

# hashes are bytes
hash_bytes = []
for i in points:
    hash_bytes.append(H_c(i))

In [5]:
# 10,000 random 32-bit K values
K_np = np.array(random.sample(range(2 ** 32), 10000), dtype=np.dtype('>u4'))
K = K_np.astype(int).tolist()

## Time reduction functions

In [ ]:
## first reduction function

# store times
r_mod_times = []

# for 10,000 trials (K)
for k in K:

    # start timer
    start_time = time.perf_counter()

    # reduce all points
    for point in hash_int:
        r(N, t, point, k)

    # stop timer
    r_time = time.perf_counter() - start_time

    # add to list
    r_mod_times.append(r_time)

# make a df 
r_df = pd.DataFrame({"mod times" : r_mod_times})

#####################################################################################

## first reduction function

# store times
r_mmh_times = []

# for 10,000 trials (K)
for k in K:

    # start timer
    start_time = time.perf_counter()

    # reduce all points
    for point in hash_bytes:
        r_mmh(N, t, point, k)

    # stop timer
    r_time = time.perf_counter() - start_time

    # add to list
    r_mmh_times.append(r_time)

# make a df 
r_df["mmh times"] = r_mmh_times

In [ ]:
## describe the dataframe
r_df.describe()

,mod times,mmh times
count,10000.000000,10000.000000
mean,0.003005,0.003545
std,0.000574,0.000626
min,0.002644,0.003066
25%,0.002721,0.003254
50%,0.002796,0.003340
75%,0.003059,0.003624
max,0.009081,0.011241


## Calculate ratio

In [ ]:
# get the mean of the hash times and reduction times
hash_int_mean = hash_df.loc[:, 'int times'].mean()
hash_bytes_mean = hash_df.loc[:, 'bytes times'].mean()

mod_mean = r_df.loc[:, 'mod times'].mean()
mmh_mean = r_df.loc[:, 'mmh times'].mean()

# print them all
print("hash_int_mean", hash_int_mean)
print("hash_bytes_mean", hash_bytes_mean)
print("mod mean", mod_mean)
print("mmh mean", mmh_mean)

# get ratio
# int_res = Fraction(hash_int_mean, mod_mean).limit_denominator
# print(f"The ratio is {int_res.numerator}:{int_res.denominator}")

# print(hash_int_mean/mod_mean)
print("Ratio of time for 1 hash v 1 reduction:")
print(f"Mod reduction function: 1 : {mod_mean/hash_int_mean}")
print(f"mmh reduction function: 1 : {mmh_mean/hash_bytes_mean}")

hash_int_mean 0.013216807279293426
hash_bytes_mean 0.010954783270228655
mod mean 0.0030050158303929495
mmh mean 0.0035448151081800463
Ratio of time for 1 hash v 1 reduction:
Mod reduction function: 1 : 0.22736321767367093
mmh reduction function: 1 : 0.3235860555830108
